<a href="https://colab.research.google.com/github/deathfire1245/3D-Space-Editor/blob/main/Blender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
drive_path = 'zane.blend' #@param {type: 'string'}
#@markdown ---
animation = False #@param {type: 'boolean'}
start_frame =  1#@param {type: 'integer'}
end_frame =  250#@param {type: 'integer'}
#@markdown ---
download_type = 'direct' #@param ['direct', 'google_drive', 'gdrive_direct'] {allow-input: false}
output_name = 'blender-##' #@param {type: 'string'}
zip_files = True #@param {type: 'boolean'}
drive_output_path = 'blender/output' #@param {type: 'string'}
#@markdown ---
gpu_enabled = True #@param {type:"boolean"}
cpu_enabled = False #@param {type:"boolean"}

In [3]:
import os
import shutil
from google.colab import files, drive

drive.mount('/drive')

Mounted at /drive


In [4]:
!mkdir render

In [5]:
!cp -r '/drive/MyDrive/{drive_path}' 'render/'

In [6]:
#find latest version 'linux-x64.tar.xz' at "https://ftp.nluug.nl/pub/graphics/blender/release/"

blender_url = "https://download.blender.org/release/Blender5.1/blender-5.1.2-linux-x64.tar.xz"
base_url = os.path.basename(blender_url)

!mkdir blender
!wget -nc $blender_url
!tar -xkf $base_url -C ./blender --strip-components=1

--2026-07-08 17:22:52--  https://download.blender.org/release/Blender5.1/blender-5.1.2-linux-x64.tar.xz
Resolving download.blender.org (download.blender.org)... 104.20.41.146, 172.66.172.236, 2606:4700:10::ac42:acec, ...
Connecting to download.blender.org (download.blender.org)|104.20.41.146|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 396237880 (378M) [application/octet-stream]
Saving to: ‘blender-5.1.2-linux-x64.tar.xz’

blender-5.1.2-linux 100%[===================>] 377.88M   256MB/s    in 1.5s    

2026-07-08 17:22:54 (256 MB/s) - ‘blender-5.1.2-linux-x64.tar.xz’ saved [396237880/396237880]



In [7]:
#Verify installation
!sudo ./blender/blender -v

Blender 5.1.2
	build date: 2026-05-19
	build time: 02:18:37
	build commit date: 2026-05-18
	build commit time: 13:43
	build hash: ec6e62d40fa9
	build branch: blender-v5.1-release
	build platform: Linux
	build type: Release
	build c flags:  -Wall -Werror=implicit-function-declaration -Wstrict-prototypes -Werror=return-type -Werror=vla -Wmissing-prototypes -Wno-char-subscripts -Wno-unknown-pragmas -Wpointer-arith -Wunused-parameter -Wwrite-strings -Wlogical-op -Wundef -Winit-self -Wmissing-include-dirs -Wno-div-by-zero -Wtype-limits -Wformat-signedness -Wrestrict -Wno-stringop-overread -Wno-stringop-overflow -Wnonnull -Wabsolute-value -Wuninitialized -Wredundant-decls -Wshadow -Wimplicit-fallthrough=5 -Wno-error=unused-but-set-variable  -march=x86-64-v2 -std=gnu11 -pipe -fPIC -funsigned-char -fno-strict-aliasing -ffp-contract=off  
	build c++ flags:  -Wuninitialized -Wredundant-decls -Wall -Wno-invalid-offsetof -Wno-sign-compare -Wlogical-op -Winit-self -Wmissing-include-dirs -Wno-div-by

In [8]:
# Enable GPU rendering (or add custom properties here)
data = "import re\n"+\
    "import bpy\n"+\
    "scene = bpy.context.scene\n"+\
    "scene.cycles.device = 'GPU'\n"+\
    "prefs = bpy.context.preferences\n"+\
    "prefs.addons['cycles'].preferences.get_devices()\n"+\
    "cprefs = prefs.addons['cycles'].preferences\n"+\
    "print(cprefs)\n"+\
    "for compute_device_type in ('CUDA', 'OPENCL', 'NONE'):\n"+\
    "    try:\n"+\
    "        cprefs.compute_device_type = compute_device_type\n"+\
    "        print('Device found:',compute_device_type)\n"+\
    "        break\n"+\
    "    except TypeError:\n"+\
    "        pass\n"+\
    "for device in cprefs.devices:\n"+\
    "    if not re.match('intel', device.name, re.I):\n"+\
    "        print('Activating',device)\n"+\
    "        device.use = "+str(gpu_enabled)+"\n"+\
    "    else:\n"+\
    "        device.use = "+str(cpu_enabled)+"\n"
with open('setgpu.py', 'w') as f:
    f.write(data)

In [9]:
!mkdir output

if not drive_output_path.endswith('/'):
    drive_output_path += '/'

if download_type != 'gdrive_direct':
    output_path = 'output/' + output_name
else:
    output_path = '/drive/My Drive/' + drive_output_path + output_name

if animation:
    if start_frame == end_frame:
        !sudo ./blender/blender -b 'render/{drive_path}' -P setgpu.py -E CYCLES -o '{output_path}' -noaudio -a
    else:
        !sudo ./blender/blender -b 'render/{drive_path}' -P setgpu.py -E CYCLES -o '{output_path}' -noaudio -s $start_frame -e $end_frame -a
else:
    !sudo ./blender/blender -b 'render/{drive_path}' -P setgpu.py -E CYCLES -o '{output_path}' -noaudio -f $start_frame

Blender 5.1.2 (hash ec6e62d40fa9 built 2026-05-19 02:18:37)
00:01.921  blend            | Read blend: "/content/render/zane.blend"
scripts disabled for "/content/render/zane.blend", skipping 'CustomRig_Vitruvian_ui.py'
00:02.481  cycles           | WARNING CUEW initialization failed: Error opening the library
00:02.482  cycles           | WARNING HIPEW initialization failed: Error opening HIP dynamic library
<bpy_struct, CyclesPreferences at 0x7a6e5728cb98>
Device found: CUDA
00:02.483  render           | Rendering single frame
00:02.483  render           | Rendering frame 1
00:02.484  render           | Start rendering: Scene, ViewLayer
00:02.484  render           | Engine: Cycles
00:02.620  render           | Fra: 1 | Mem: 0M | Synchronizing object | Light
00:19.144  render           | Fra: 1 | Mem: 0M | Synchronizing object | cm_vitruvian
00:19.437  render           | Fra: 1 | Mem: 0M | Initializing
00:19.586  render           | Fra: 1 | Mem: 0M | Waiting for render to start
00:19.5

Simple export:

In [10]:
!cp -r 'output/' '/drive/MyDrive/'

Other exports:

In [11]:
path, dirs, files_folder = next(os.walk("output"))
output_folder_name = output_name.replace('#', '') + 'render'

if download_type == 'gdrive_direct':
    pass
elif len(files_folder) == 1:
    render_img = 'output/' + files_folder[0]
    if download_type == 'direct':
        files.download('output/' + files_folder[0])
    else:
        shutil.copy('/content/' + render_img, '/drive/My Drive/' + drive_output_path)
elif len(files_folder) > 1:
    if zip_files:
        shutil.make_archive(output_folder_name, 'zip', 'output')
    if download_type == 'direct':
        files.download(output_folder_name + '.zip')
    else:
        shutil.copy('/content/' + output_folder_name + ".zip", '/drive/My Drive/' + drive_output_path)
elif download_type == 'direct':
    for f in files_folder:
        files.download('output/{}'.format(f))
    # Drive, no zip
    else:
        for f in files_folder:
          shutil.copy("/content/output/" + f, '/drive/My Drive/' + drive_output_path + f)
else:
    raise SystemExit("No frames are rendered.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Disclaimer
The GPU used in Google Colab is specialized for data centres, neural network etc, not rendering 3D scenes. Because the computing power provided are free, the usage limits, idle timeouts and speed of the rendering may varies. [ColabPro](https://colab.research.google.com/signup) is available for those who wanted to have more powerful GPU and longer session for rendering. See the [FAQ](https://research.google.com/colaboratory/faq.html) for more info about this platform. In some cases, it might be faster to use a renderfarm such as [Sheepit](https://www.sheepit-renderfarm.com/) (free) and [ConciergeRender](https://www.conciergerender.com/) (paid w/ trial) which support parallel rendering.

## License
```
MIT License

Copyright (c) 2020 syn73

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```